# Exploring a jitter sweep

Reads the JSON written by `python main.py experiment=jitter_sweep` and rebuilds the three
tables of [`docs/jitter_sweep.md`](../docs/jitter_sweep.md): how far the perturbation moves the
cloud, what each method costs, and how often its in-sphere test falls back to exact arithmetic.

Nothing here recomputes a triangulation, so it runs on a laptop with only numpy installed. Point
`RESULTS` at a run of your own; `results/` is not tracked by git, so a fresh clone has none until
a job has been run.

In [ ]:
import json
import os

RESULTS = "../results/jitter/jitter.json"

if not os.path.exists(RESULTS):
    raise SystemExit(
        f"no results at {RESULTS}.\n"
        "Run `python main.py experiment=jitter_sweep` (a GPU node), or set RESULTS to a\n"
        "jitter.json you already have."
    )

with open(RESULTS) as fh:
    data = json.load(fh)

runs = {(r["cloud"], r["jitter"], r["method"]): r for r in data["runs"]}
preds = {(p["cloud"], p["jitter"], p["method"]): p for p in data["predicates"]}
defos = {(d["cloud"], d["jitter"]): d for d in data["deformations"]}

clouds = sorted({c for c, _ in defos})
jitters = sorted({j for _, j in defos})
print(data["_env"].get("date"), "|", data["_env"].get("gpu", "no GPU recorded"))
print(f"{len(runs)} measurements, {len(preds)} counter sets, clouds: {', '.join(clouds)}")

## How much the jitter deforms the cloud

The displacement only means something relative to the distance between neighbouring points. The
last three columns are about the triangulation rather than the cloud.

In [ ]:
for cloud in clouds:
    print(f"\n{cloud}")
    print(
        f"{'jitter':>8} {'shift/spacing':>14} {'hull vol':>10} {'nn changed':>11}"
        f" {'tets':>9} {'slivers':>8} {'near-flat':>10}"
    )
    for j in jitters:
        d = defos[(cloud, j)]
        r = d["reference"]
        print(
            f"{j:>8g} {100 * (d['displacement_over_spacing'] or 0):>13.3f}%"
            f" {100 * d.get('hull_volume_rel_change', 0):>9.4f}%"
            f" {100 * d['nearest_neighbour_changed']:>10.2f}%"
            f" {r['tets']:>9} {r['slivers']:>8} {r['degenerate_tets']:>10}"
        )

## In-sphere tests that needed exact arithmetic

Only the methods that can report it appear: gDel3D, CGAL and the Paragram converter. A method
with no exact fallback to count is absent rather than shown as zero.

In [ ]:
methods = sorted({m for _, _, m in preds})
for cloud in clouds:
    print(f"\n{cloud}")
    print(f"{'jitter':>8} " + " ".join(f"{m:>28}" for m in methods))
    for j in jitters:
        cells = []
        for m in methods:
            p = preds.get((cloud, j, m))
            cells.append(
                f"{'-':>28}"
                if p is None
                else f"{p['exact']:>9}/{p['total']:<10} {100 * p['exact'] / max(1, p['total']):>6.3f}%"
            )
        print(f"{j:>8g} " + " ".join(cells))

## Time, and whether the answer is right

`identical` means the method returned exactly the same set of tetrahedra as the CGAL reference
computed on the same points; otherwise the counts are missing / extra.

In [ ]:
all_methods = sorted({m for _, _, m in runs})
for cloud in clouds:
    print(f"\n{cloud}   seconds (and agreement with CGAL)")
    print(f"{'method':>18} " + " ".join(f"{j:>18g}" for j in jitters))
    for m in all_methods:
        cells = []
        for j in jitters:
            r = runs.get((cloud, j, m))
            if r is None or r.get("status") != "ok":
                cells.append(f"{(r or {}).get('status', '-'):>18}")
                continue
            c = r.get("compare") or {}
            miss, extra = c.get("ref_only", 0), c.get("method_only", 0)
            agree = "identical" if not miss and not extra else f"-{miss}/+{extra}"
            cells.append(f"{r['seconds']:>7.3f} {agree:>10}")
        print(f"{m:>18} " + " ".join(cells))